# DV01: Interest Rate Risk Sensitivity on a Treasury Fixed-Income Portfolio

**Context.** A treasury desk manages more than FX exposures — it typically also holds interest-rate-sensitive
positions (government notes, fixed-rate loans) as part of its liquidity and investment book. **DV01** is the
standard metric for measuring and hedging that risk. Unlike Value at Risk (see the companion `VaR99`
notebook), which measures the probability-weighted loss from many risk factors moving together, DV01
measures something narrower and far more actionable: how much a portfolio's value changes from one specific,
small move — a **1 basis point (0.01%) change in interest rates**. That narrow focus is exactly what makes
DV01 the natural tool for sizing a hedge.

**Goal.** Given a small, hypothetical treasury fixed-income portfolio (government notes held as part of a
liquidity/investment book, the kind of position a treasury desk holds alongside its FX exposures), we will:

1. Compute each position's DV01 using the standard **bump-and-reprice** method.
2. Cross-check it against the closed-form **modified duration** formula.
3. Use the resulting portfolio DV01 to **size an interest-rate hedge** — turning a risk number into a
   concrete hedging decision.

## 1. What is DV01?

**DV01 ("Dollar Value of a 01", where "01" means one basis point, 0.01%) answers a very concrete question:
if interest rates move up by exactly 1 basis point, how much money do I gain or lose?**

Formally, it is a derivative of price with respect to yield:

$$
\text{DV01} = -\frac{\partial P}{\partial y} \times 0.0001
$$

where $P$ is the price of the instrument and $y$ is its **yield** — the single discount rate that, applied
to all its future cash flows (Section 4), gives back its current price. It is the market's compressed way of
quoting "the annualised return this bond offers at its current price."

The minus sign is there so that DV01 comes
out **positive** for a typical bond, whose price falls when yields rise.

**In practice, nobody differentiates by hand.** The standard method — used throughout this notebook — is to
*reprice* the instrument at today's yield, reprice it again after bumping the yield up by exactly 1bp, and
take the (negated) difference:

$$
\text{DV01} \approx -\big[P(y + 0.0001) - P(y)\big]
$$

**How this differs from VaR.** VaR asks a *probabilistic* question — "how much could I lose, with 99%
confidence, over the next day, from any combination of market moves?" — and needs a full covariance matrix
of returns to answer it. DV01 asks a much narrower, *deterministic* question: "how much do I lose from one
specific, small move in one specific risk factor (interest rates)?"

## 2. Data source

We use the current 10-year US Treasury yield, pulled from Yahoo Finance (`^TNX`, quoted as yield×10 — e.g.
47.0 means 4.70%), as a realistic base interest rate level for the example. A full analysis would use the
entire yield curve (a different rate for every maturity); here we simplify by pricing each note off its own
single, fixed yield, flat over its life.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams["figure.figsize"] = (10, 4)

In [2]:
tnx = yf.download("^TNX", period="5d", progress=False)["Close"]
current_10y_yield = tnx.iloc[-1].item() / 100  # ^TNX is quoted as yield x 10

print(f"Current 10Y US Treasury yield: {current_10y_yield:.4%}")

Current 10Y US Treasury yield: 4.7000%


## 3. A hypothetical treasury fixed-income portfolio

As part of managing its cash reserves, a treasury desk typically invests a portion of its liquidity in
low-risk, interest-rate-sensitive instruments such as government notes. A **note** is simply a government
bond with an original maturity of up to 10 years (the US Treasury reserves "bond" for longer maturities) —
it pays a fixed coupon on a regular schedule and returns the face value at maturity, exactly like the
generic bond priced in Section 4. We define a small illustrative portfolio of three USD-denominated
government notes of different maturities:

| Position | Face value | Coupon | Maturity | Yield |
|---|---|---|---|---|
| 2Y Note  | 20,000,000 | 4.00% | 2 years  | current 10Y yield − 0.30% |
| 5Y Note  | 15,000,000 | 4.30% | 5 years  | current 10Y yield − 0.10% |
| 10Y Note | 10,000,000 | 4.70% | 10 years | current 10Y yield |

(Coupons and yields are set close to — but not identical to — the current 10-year yield pulled above, to
give the portfolio a realistic-looking yield curve shape: shorter maturities yielding a little less than
longer ones, as is typical outside of an inverted curve.)

In [3]:
portfolio = [
    {"name": "2Y Note",  "face": 20_000_000, "coupon": 0.040, "maturity": 2,  "yield": current_10y_yield - 0.0030},
    {"name": "5Y Note",  "face": 15_000_000, "coupon": 0.043, "maturity": 5,  "yield": current_10y_yield - 0.0010},
    {"name": "10Y Note", "face": 10_000_000, "coupon": 0.047, "maturity": 10, "yield": current_10y_yield},
]

pd.DataFrame(portfolio).set_index("name")

,face,coupon,maturity,yield
name,,,,
2Y Note,20000000,0.040,2,0.044
5Y Note,15000000,0.043,5,0.046
10Y Note,10000000,0.047,10,0.047


## 4. Bond pricing

A fixed-coupon bond's price is the present value of all its future cash flows (coupons plus the face value
at maturity), discounted at its yield $y$:

$$
P = \sum_{i=1}^{n} \frac{C/f}{(1+y/f)^i} \;+\; \frac{F}{(1+y/f)^n}
$$

where $F$ is the face value, $C$ is the annual coupon rate, $f$ is the number of coupon payments per year
(we use $f=2$, semi-annual — the standard market convention for government bonds), and $n = \text{maturity}
\times f$ is the total number of coupon periods.

In [4]:
def bond_price(face, coupon_rate, maturity_years, yield_rate, freq=2):
    coupon = face * coupon_rate / freq
    n = int(maturity_years * freq)
    periods = np.arange(1, n + 1)
    discount_factors = 1 / (1 + yield_rate / freq) ** periods
    return np.sum(coupon * discount_factors) + face * discount_factors[-1]


for bond in portfolio:
    bond["price"] = bond_price(bond["face"], bond["coupon"], bond["maturity"], bond["yield"])

pd.DataFrame(portfolio).set_index("name")[["face", "coupon", "maturity", "yield", "price"]]

,face,coupon,maturity,yield,price
name,,,,,
2Y Note,20000000,0.040,2,0.044,1.984843e+07
5Y Note,15000000,0.043,5,0.046,1.480103e+07
10Y Note,10000000,0.047,10,0.047,1.000000e+07


## 5. DV01 via bump-and-reprice

Applying the formula from Section 1 to each note: price it at its current yield, price it again 1bp higher,
and take the (negated) difference.

**DV01 is additive.** Just like the dollar exposures $\mathbf{e}$ in the VaR notebook, DV01 stacks across
positions: the portfolio's total DV01 is simply the sum of each position's DV01. This additivity is exactly
what makes DV01 so useful for hedging — if you know every instrument's DV01, you can always find the
combination of positions whose DV01s cancel out.

In [5]:
def dv01_bump(face, coupon_rate, maturity_years, yield_rate, freq=2, bump=0.0001):
    p0 = bond_price(face, coupon_rate, maturity_years, yield_rate, freq)
    p1 = bond_price(face, coupon_rate, maturity_years, yield_rate + bump, freq)
    return -(p1 - p0)


for bond in portfolio:
    bond["dv01"] = dv01_bump(bond["face"], bond["coupon"], bond["maturity"], bond["yield"])

dv01_df = pd.DataFrame(portfolio).set_index("name")[["face", "price", "dv01"]]
total_dv01 = dv01_df["dv01"].sum()

print(dv01_df.round(2))
print(f"\nTotal portfolio DV01: ${total_dv01:,.2f} per basis point")

              face        price     dv01
name                                    
2Y Note   20000000  19848427.27  3770.53
5Y Note   15000000  14801027.90  6579.03
10Y Note  10000000  10000000.15  7902.41

Total portfolio DV01: $18,251.97 per basis point


That number means: if interest rates rise by 1 basis point (0.01%) across the board, this portfolio loses
approximately that many dollars. If rates rise by 10bp instead, the loss scales up roughly linearly (to
about 10× the DV01) — this linear approximation is only reliable for small moves, a point we come back to
in Section 8.

## 6. Cross-check: DV01 via Modified Duration

DV01 also has a closed-form relationship to **modified duration**, a more traditional (if less intuitive)
interest-rate risk measure. Modified duration is the price sensitivity expressed in *percentage* terms;
DV01 simply re-expresses that in *dollar* terms:

$$
\text{DV01} = \text{Modified Duration} \times P \times 0.0001
$$

We compute modified duration analytically — as the weighted-average time to each cash flow, adjusted for
compounding — and confirm it gives (almost) the same DV01 as the bump-and-reprice method above. The tiny
remaining difference is just the numerical approximation of a derivative using a finite 1bp step.

In [6]:
def modified_duration(face, coupon_rate, maturity_years, yield_rate, freq=2):
    coupon = face * coupon_rate / freq
    n = int(maturity_years * freq)
    periods = np.arange(1, n + 1)
    cashflows = np.full(n, coupon)
    cashflows[-1] += face
    discount_factors = 1 / (1 + yield_rate / freq) ** periods
    pv_cashflows = cashflows * discount_factors
    price = pv_cashflows.sum()
    macaulay_duration = np.sum((periods / freq) * pv_cashflows) / price
    return macaulay_duration / (1 + yield_rate / freq), price


bond = portfolio[-1]  # 10Y Note, as an example
mod_duration, price_check = modified_duration(bond["face"], bond["coupon"], bond["maturity"], bond["yield"])
dv01_analytic = mod_duration * price_check * 0.0001

print(f"{bond['name']}: modified duration = {mod_duration:.3f} years")
print(f"DV01 (bump-and-reprice): ${bond['dv01']:,.2f}")
print(f"DV01 (modified duration formula): ${dv01_analytic:,.2f}")

10Y Note: modified duration = 7.906 years
DV01 (bump-and-reprice): $7,902.41
DV01 (modified duration formula): $7,906.17


## 7. Using DV01 to size a hedge

This is the practical payoff: DV01's main use in a treasury function is sizing a hedge.

Suppose the desk wants to neutralise the portfolio's interest-rate exposure using a **10-year interest rate
swap** (approximated here, for simplicity, as if it were a bond of matching maturity and coupon priced at
the current market yield — real swap pricing involves a floating leg and its own discount curve, but its
DV01 behaves the same way for this purpose).

To hedge, we need a position whose DV01 exactly offsets the portfolio's DV01 — same size, opposite sign:

$$
\text{Hedge notional} = \frac{\text{Portfolio DV01}}{\text{DV01 per 100 of hedge instrument}} \times 100
$$

This is exactly the calculation that would sit behind an automated hedging strategy: recompute the
portfolio's DV01 whenever positions change, and automatically size — and potentially execute — the
offsetting swap trade.

In [7]:
hedge_coupon = 0.047
hedge_yield = current_10y_yield
hedge_maturity = 10

hedge_dv01_per_100 = dv01_bump(100, hedge_coupon, hedge_maturity, hedge_yield)
hedge_notional = total_dv01 / hedge_dv01_per_100 * 100

print(f"Hedge instrument (10Y swap-equivalent) DV01 per 100 notional: ${hedge_dv01_per_100:.4f}")
print(f"Hedge notional needed: ${hedge_notional:,.0f} (pay-fixed / short position of this size)")

Hedge instrument (10Y swap-equivalent) DV01 per 100 notional: $0.0790
Hedge notional needed: $23,096,704 (pay-fixed / short position of this size)


## 8. Conclusions and limitations

- DV01 is a **linear approximation** valid only for small moves. It assumes a **parallel shift**: every
  maturity's yield moves by the same 1bp. For larger moves, the true price change curves away from this
  linear estimate — the correction for that curvature is called **convexity**, a natural next step beyond
  DV01.
- We priced each note off a **single flat yield**. Real desks compute DV01 **per point on the yield curve**
  (a "key rate DV01" for the 2Y point, the 5Y point, the 10Y point, etc.), because real yield curves rarely
  move in a perfect parallel shift — short and long rates can move by different amounts, or even in opposite
  directions.
- DV01 and VaR are **complementary, not competing** metrics. VaR answers "how much could this portfolio
  lose overall, with some probability, from any combination of market moves?" — it needs a covariance
  matrix and a distributional assumption. DV01 answers a narrower question — "how much would this portfolio
  lose from one specific, small move in one specific risk factor?" — and that narrower, deterministic
  answer is exactly what makes it simple to turn directly into a hedge size, as in Section 7.
- In a treasury function, this is the standard workflow for managing interest-rate risk: compute each
  position's DV01, aggregate it across the book, and use that number to size the swap or future that
  neutralises the exposure — ideally as an automated, repeatable calculation rather than a one-off
  spreadsheet exercise.